# MMTFv3 CL+GC Optuna — PTP Continuous Position Sizing

Trains **MMTFv3Core** (Mamba or Transformer backbone, Optuna-selected) across CL, GC
using **PredictionToPosition (PTP)** — a 5-class quantile head that converts
confidence-weighted softmax probabilities into continuous positions in [-1, 1].

## Model: MMTFv3Core + QuantilePositionHead
6-phase architecture:
1. **VAE Regime Encoder**: Daily [ret_1d, ret_5d, ret_21d, rv_1d] -> z_regime
2. **Static Context**: Identity + z_regime -> c_s, c_e, c_c, c_h
3. **Technical Backbone**: ContinuousIntradayPrep features (incl. tod_sin) -> Mamba/Transformer
4. **Cross-Modal Branches**: Spatial (NumberBars + VPIN raster) + Sequential (tabular VPIN)
5. **Regime-Conditioned BVS**: Weighted branch selection
6. **Enrichment + Temporal Attention**: TFT-style attention -> **QuantilePositionHead** -> context-aware PTP -> position

## 5-Class Conviction Levels
- **0**: Strong negative -> position ~ -1.0
- **1**: Weak negative -> position ~ -0.5
- **2**: Neutral -> position ~ 0.0
- **3**: Weak positive -> position ~ +0.5
- **4**: Strong positive -> position ~ +1.0

## Loss + Scheduling
```
total = ce_weight * ProfitWeightedCE(logits, class_labels)
      + pnl_weight * DownsideAwareTradingLoss(position, returns)
      + recon_weight * ae_reconstruction
```
- `SharpeScheduler` still shifts direction -> Sharpe emphasis over training.
- `HeadAwarePTPScheduler` separately controls the quantile head and PTP mapper lrs.
- Class labels are derived from sigma-based return bucketing per batch.
- AE hidden context and temporal context are fed directly into the PTP mapping.

## Target
30-minute forward log return from 5-min intraday bars.

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow
    !git pull
    %cd ..
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow and optuna are installed")

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Configuration

In [ ]:
# --- Tickers ---
TICKERS = ['CL', 'GC']

# --- Architecture ---
TUNE_BACKBONE = True     # Let Optuna choose mamba/transformer
BACKBONE = 'mamba'       # Default if TUNE_BACKBONE=False
USE_PTP = True           # Use PredictionToPosition (5-class quantile head)

# --- Target ---
TARGET_HORIZON_MINUTES = 60   # 60min forward return (was 30 — longer to reduce TC drag)
BAR_MINUTES = 5               # 5min bars → target = y_fwd_12

# --- Session Filter ---
# Options:
#   SAMPLE_SESSION = "usa"                     # Named: "usa", "london", "overlap"
#   SAMPLE_SESSION = None                      # Use all active session bars
#   SAMPLE_SESSION_START/END = "09:30"/"15:00" # Custom time window
SAMPLE_SESSION = "usa"              # Filter to USA session only
SAMPLE_SESSION_START = None         # Custom start (overrides SAMPLE_SESSION if both set)
SAMPLE_SESSION_END = None           # Custom end

# --- Stride ---
# stride=12 with 5min bars + 60min target → non-overlapping samples
# stride=6 → 50% overlap, stride=1 → every bar (max overlap)
SAMPLE_STRIDE = 12

# --- AE ---
AE_WINDOW = 21                # 21 trading days lookback for regime VAE
F_AE = 4                      # [ret_1d, ret_5d, ret_21d, rv_1d]

# --- Paths ---
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_ROOT = DRIVE_PATH / 'features'
    RESULTS_PATH = DRIVE_PATH / 'results' / 'mmtfv3_cl_gc'
else:
    DATA_ROOT = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/mmtfv3_cl_gc')

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"Tickers: {TICKERS}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return")
if SAMPLE_SESSION_START and SAMPLE_SESSION_END:
    print(f"Session filter: custom {SAMPLE_SESSION_START}-{SAMPLE_SESSION_END}")
elif SAMPLE_SESSION:
    print(f"Session filter: {SAMPLE_SESSION.upper()}")
else:
    print(f"Session filter: all active bars")
print(f"Stride: {SAMPLE_STRIDE} bars ({SAMPLE_STRIDE * BAR_MINUTES}min between samples)")
print(f"AE window: {AE_WINDOW} days, f_ae={F_AE}")
print(f"PTP (5-class quantile head): {'ENABLED' if USE_PTP else 'DISABLED'}")
if TUNE_BACKBONE:
    print(f"Backbone: Optuna-tuned (mamba / transformer)")
else:
    print(f"Backbone: {BACKBONE.upper()} (fixed)")

In [ ]:
# Verify data files for each ticker
print("Checking data files...")
required_files = ['intraday.csv', 'profiles.npz', 'rasterized.npz', 'vpin.parquet']

all_found = True
for ticker in TICKERS:
    ticker_path = DATA_ROOT / ticker
    print(f"\n{ticker}:")
    for fname in required_files:
        fpath = ticker_path / fname
        status = "[OK]" if fpath.exists() else "[MISSING]"
        print(f"  {status} {fname}")
        if not fpath.exists():
            all_found = False

if not all_found:
    print("\n[WARNING] Some files missing - data loading may fail")

## 3. Load Data via V3ContinuousPrep

In [ ]:
from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    v3_collate_fn,
    unpack_v3_batch,
    build_v3_loaders,
)
from CTAFlow.models.prep.intraday_continuous import SessionSpec

print("Loading ticker data...")
prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec("USA", "08:30", "16:00")],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)

dims = prep.get_dims()
print(f"\nFeature dimensions: {dims}")
print(f"Tech feature columns ({dims['f_tech']}): {prep._tech_feature_cols[:10]}...")
print(f"n_tickers: {prep.n_tickers}")
print(f"n_asset_classes: {prep.n_asset_classes}")
print(f"n_asset_subclasses: {prep.n_asset_subclasses}")

In [ ]:
# Verify tech features include tod_sin
assert 'tod_sin' in prep._tech_feature_cols, "tod_sin missing from feature cols!"
print("tod_sin confirmed in feature columns")

# Quick target distribution check
fig, axes = plt.subplots(1, len(TICKERS), figsize=(6*len(TICKERS), 4))
if len(TICKERS) == 1:
    axes = [axes]

for ticker, ax in zip(TICKERS, axes):
    df = prep._tech_dfs.get(ticker)
    if df is None:
        continue
    target = df[prep._target_col].dropna()
    ax.hist(target.values, bins=100, alpha=0.7, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{ticker} Target ({prep._target_col})')
    ax.set_xlabel('30min Forward Log Return')
    ax.set_ylabel('Count')
    print(f"{ticker}: mean={target.mean():.6f}, std={target.std():.6f}, n={len(target)}")

plt.tight_layout()
plt.show()

## 4. Define Optuna Objective

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import (
    MMTFv3Core,
    MMTFv3Mamba,
    MMTFv3Transformer,
    StatefulMMTFv3Core,
    TickerPositionStateLayer,
    PTPLoss,
    HeadAwarePTPScheduler,
    build_ptp_optimizer_param_groups,
    returns_to_classes,
    train_epoch_v3,
    train_epoch_v3_stateful,
    train_epoch_v3_ptp,
    evaluate_v3,
    evaluate_v3_stateful,
    evaluate_v3_ptp,
    ContinuousTradingLoss,
    SharpeScheduler,
)

F_TECH = dims['f_tech']
F_SEQ = dims['f_seq']
NUMBARS_CHANNELS = dims['numbars_channels']
VPIN_TIME = dims['vpin_time']
VPIN_CHANNELS = dims['vpin_channels']
VPIN_BINS = dims['vpin_bins']

print(f"Model input dimensions:")
print(f"  f_tech={F_TECH} (ContinuousIntradayPrep features incl. tod_sin)")
print(f"  f_seq={F_SEQ} (tabular VPIN, scaled)")
print(f"  f_ae={F_AE} (daily returns: ret_1d, ret_5d, ret_21d, rv_1d, z-scored)")
print(f"  numbars_channels={NUMBARS_CHANNELS}")
print(f"  vpin_raster: time={VPIN_TIME}, channels={VPIN_CHANNELS}, bins={VPIN_BINS}")
print(f"  n_tickers={prep.n_tickers}, n_asset_classes={prep.n_asset_classes}, n_subclasses={prep.n_asset_subclasses}")
print(f"\nArchitecture: MMTFv3Core + QuantilePositionHead + StateLayer" if USE_PTP else "\nArchitecture: MMTFv3Core + StateLayer")
print(f"Loss: PTPLoss (CE + downside-aware PnL + AE)" if USE_PTP else "Loss: ContinuousTradingLoss + SharpeScheduler")


In [ ]:
def objective(trial: optuna.Trial) -> float:
    # --- Backbone selection ---
    if TUNE_BACKBONE:
        backbone = trial.suggest_categorical('backbone', ['mamba', 'transformer'])
    else:
        backbone = BACKBONE

    # --- Shared architecture ---
    d_model = trial.suggest_categorical('d_model', [64, 128])
    d_static_emb = trial.suggest_categorical('d_static_emb', [32, 64])
    n_heads = trial.suggest_categorical('n_heads', [2, 4])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    grn_dropout = trial.suggest_float('grn_dropout', 0.05, 0.4)

    # --- Position state layer ---
    state_hidden_dim = trial.suggest_categorical('state_hidden_dim', [8, 16, 32])
    state_momentum = trial.suggest_float('state_momentum', 0.90, 0.99)  # bias toward stickier positions

    # --- PTP parameters ---
    if USE_PTP:
        ptp_temperature = trial.suggest_float('ptp_temperature', 1.0, 3.0)
        ce_weight = trial.suggest_float('ce_weight', 0.3, 2.0, log=True)
        pnl_weight = trial.suggest_float('pnl_weight', 0.3, 2.0, log=True)
        profit_scale = trial.suggest_float('profit_scale', 50.0, 200.0)
        inner_threshold = trial.suggest_float('inner_threshold', 0.15, 0.4)
        outer_threshold = trial.suggest_float('outer_threshold', 0.8, 1.5)
        quantile_lr_scale = trial.suggest_float('quantile_lr_scale', 0.75, 1.35)
        ptp_lr_scale = trial.suggest_float('ptp_lr_scale', 0.50, 1.25)
        head_lr_decay = trial.suggest_float('head_lr_decay', 0.45, 0.75)
        quantile_patience = trial.suggest_int('quantile_patience', 1, 3)
        ptp_patience = trial.suggest_int('ptp_patience', 1, 3)
        head_overfit_tolerance = trial.suggest_float('head_overfit_tolerance', 0.02, 0.12)

    # --- Backbone-specific ---
    if backbone == 'mamba':
        d_state = trial.suggest_categorical('d_state', [16, 32])
        d_conv = trial.suggest_categorical('d_conv', [2, 4])
        expand = trial.suggest_categorical('expand', [1, 2])
        n_layers = trial.suggest_int('n_layers', 1, 2)
        d_ff = 512
    else:
        n_layers = trial.suggest_int('n_layers', 2, 4)
        d_ff = trial.suggest_categorical('d_ff', [256, 512])
        d_state, d_conv, expand = 16, 4, 2

    # --- VAE ---
    ae_type = trial.suggest_categorical('ae_type', ['vae', 'vqvae'])
    d_latent = trial.suggest_categorical('d_latent', [32, 64])
    d_ae_hidden = trial.suggest_categorical('d_ae_hidden', [64, 128])
    kl_weight = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)

    # --- Training ---
    tech_lookback = trial.suggest_categorical('tech_lookback', [32, 64, 96])
    seq_lookback = trial.suggest_categorical('seq_lookback', [6, 12, 24])
    batch_size = trial.suggest_categorical('batch_size', [48, 64, 96])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float('max_norm', 0.5, 1.0)

    # --- ContinuousTradingLoss (used inside PTPLoss or standalone) ---
    tc_cost = trial.suggest_float('tc_cost', 5e-5, 5e-4, log=True)  # calibrated to real ~0.5bps
    init_direction_weight = trial.suggest_float('init_direction_weight', 0.5, 1.5)
    final_direction_weight = trial.suggest_float('final_direction_weight', 0.05, 0.3)
    init_reg_weight = trial.suggest_float('init_reg_weight', 0.1, 0.5)
    target_exposure = trial.suggest_float('target_exposure', 0.2, 0.5)
    use_sortino = trial.suggest_categorical('use_sortino', [True])
    downside_vol_weight = trial.suggest_float('downside_vol_weight', 0.02, 0.75, log=True)
    holding_weight = trial.suggest_float('holding_weight', 0.0, 0.5)
    tc_in_sharpe = True  # embed TC drag into Sharpe objective

    NUM_EPOCHS = 20
    WARMUP_EPOCHS = 5

    # --- Dataloaders (with session filter + stride) ---
    try:
        train_loader, val_loader = build_v3_loaders(
            prep,
            tech_lookback=tech_lookback,
            seq_lookback_bars=seq_lookback,
            batch_size=batch_size,
            sample_session=SAMPLE_SESSION,
            sample_session_start=SAMPLE_SESSION_START,
            sample_session_end=SAMPLE_SESSION_END,
            stride=SAMPLE_STRIDE,
        )
    except Exception as e:
        print(f"Dataloader failed: {e}")
        return -1e9

    # --- Model (with correct spatial dims from data) ---
    model_kwargs = dict(
        f_tech=F_TECH,
        f_seq=F_SEQ,
        f_ae=F_AE,
        ae_type=ae_type,
        d_latent=d_latent,
        d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight,
        recon_weight=recon_weight,
        n_tickers=prep.n_tickers,
        n_asset_classes=prep.n_asset_classes,
        n_asset_subclasses=prep.n_asset_subclasses,
        d_model=d_model,
        d_static_emb=d_static_emb,
        backbone=backbone,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        dropout=dropout,
        grn_dropout=grn_dropout,
        numbars_channels=NUMBARS_CHANNELS,
        vpin_channels=VPIN_CHANNELS,
        vpin_bins=VPIN_BINS,
        vpin_time=VPIN_TIME,
    )

    base_model = MMTFv3Core(**model_kwargs)
    model = StatefulMMTFv3Core(
        base_model=base_model,
        n_tickers=prep.n_tickers,
        quantile_head=USE_PTP,
        ptp_temperature=ptp_temperature if USE_PTP else 1.5,
        state_hidden_dim=state_hidden_dim,
        state_momentum=state_momentum,
        update_on_eval=True,
    ).to(device)

    # --- Loss ---
    trading_kwargs = dict(
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        use_sortino=use_sortino,
        downside_vol_weight=downside_vol_weight,
        tc_in_sharpe=tc_in_sharpe,
        holding_weight=holding_weight,
    )

    if USE_PTP:
        loss_fn = PTPLoss(
            ce_weight=ce_weight,
            pnl_weight=pnl_weight,
            profit_scale=profit_scale,
            inner_threshold=inner_threshold,
            outer_threshold=outer_threshold,
            **trading_kwargs,
        ).to(device)
        _train_fn = train_epoch_v3_ptp
        _eval_fn = evaluate_v3_ptp
        optimizer = optim.AdamW(
            build_ptp_optimizer_param_groups(
                model,
                base_lr=learning_rate,
                weight_decay=weight_decay,
                quantile_lr_scale=quantile_lr_scale,
                ptp_lr_scale=ptp_lr_scale,
            )
        )
        scheduler = HeadAwarePTPScheduler(
            optimizer=optimizer,
            total_epochs=NUM_EPOCHS,
            head_decay=head_lr_decay,
            quantile_patience=quantile_patience,
            ptp_patience=ptp_patience,
            overfit_tolerance=head_overfit_tolerance,
            ptp_downside_weight=downside_vol_weight,
        )
    else:
        loss_fn = ContinuousTradingLoss(**trading_kwargs).to(device)
        _train_fn = train_epoch_v3_stateful
        _eval_fn = evaluate_v3_stateful
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    sharpe_sched = SharpeScheduler(
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        initial_direction_weight=init_direction_weight,
        final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight,
        final_reg_weight=0.05,
        initial_target_exposure=0.2,
        final_target_exposure=target_exposure,
        initial_holding_weight=0.0,
        final_holding_weight=holding_weight,
    )

    best_sharpe = -1e9
    patience_counter = 0
    prev_val_loss = None

    # Get inner trading loss for SharpeScheduler
    _inner_trading_loss = loss_fn.trading_loss if USE_PTP else loss_fn

    for epoch in range(NUM_EPOCHS):
        sharpe_sched.step(epoch, _inner_trading_loss)

        train_loss, train_metrics = _train_fn(
            model, train_loader, loss_fn, optimizer, device,
            max_norm=max_norm, unpack_fn=unpack_v3_batch,
        )
        val_metrics = _eval_fn(
            model, val_loader, loss_fn, device,
            unpack_fn=unpack_v3_batch,
        )

        if USE_PTP:
            lr_state = scheduler.step(
                epoch,
                {
                    'train_ce_loss': train_metrics.get('ce_loss'),
                    'val_ce_loss': val_metrics.get('ce_loss'),
                    'train_trading_loss': train_metrics.get('trading_loss'),
                    'val_trading_loss': val_metrics.get('trading_loss'),
                    'train_downside_vol': train_metrics.get('downside_vol', 0.0),
                    'val_downside_vol': val_metrics.get('downside_vol', 0.0),
                },
            )
        else:
            scheduler.step()
            lr_state = {'trunk': optimizer.param_groups[0]['lr']}

        val_loss = val_metrics['loss']
        val_sharpe = val_metrics['sharpe']

        if math.isnan(val_loss) or math.isinf(val_loss):
            print(f"  E{epoch+1:02d} | val_loss NaN/Inf -- killing trial")
            return -1e9
        if val_loss > 100.0:
            print(f"  E{epoch+1:02d} | val_loss={val_loss:.1f} exploding -- killing trial")
            return -1e9
        if prev_val_loss is not None and val_loss > prev_val_loss * 5.0 and epoch >= 3:
            print(f"  E{epoch+1:02d} | val_loss spiked -- killing trial")
            return -1e9
        prev_val_loss = val_loss

        cls_info = ""
        if USE_PTP and 'cls_accuracy' in val_metrics:
            cls_info = (
                f" | ClsAcc: {val_metrics['cls_accuracy']:.1f}%"
                f" | QLR: {lr_state.get('quantile_head', lr_state.get('trunk', 0.0)):.2e}"
                f" | PLR: {lr_state.get('ptp_head', 0.0):.2e}"
                f" | DVol: {val_metrics.get('downside_vol', 0.0):.4f}"
            )

        print(
            f"  E{epoch+1:02d} | Loss: {val_loss:.4f} | Sharpe: {val_sharpe:.4f} "
            f"| WinRate: {val_metrics['win_rate']:.1f}% | DirAcc: {val_metrics['dir_accuracy']:.1f}% "
            f"| Exposure: {val_metrics['avg_exposure']:.3f}{cls_info} | {backbone.upper()}"
        )

        if val_sharpe > best_sharpe:
            best_sharpe = val_sharpe
            patience_counter = 0
            trial.set_user_attr('final_sharpe', val_sharpe)
            trial.set_user_attr('final_sortino', val_metrics['sortino'])
            trial.set_user_attr('final_win_rate', val_metrics['win_rate'])
            trial.set_user_attr('final_dir_acc', val_metrics['dir_accuracy'])
            trial.set_user_attr('final_pf', val_metrics['profit_factor'])
            trial.set_user_attr('final_exposure', val_metrics['avg_exposure'])
            trial.set_user_attr('final_loss', val_loss)
            trial.set_user_attr('final_downside_vol', val_metrics.get('downside_vol', 0.0))
            trial.set_user_attr('backbone', backbone)
            trial.set_user_attr('state_momentum', state_momentum)
            trial.set_user_attr('state_hidden_dim', state_hidden_dim)
            if USE_PTP:
                trial.set_user_attr('final_cls_acc', val_metrics.get('cls_accuracy', 0))
                trial.set_user_attr('final_cls_dir_acc', val_metrics.get('cls_dir_accuracy', 0))
        else:
            patience_counter += 1

        trial.report(val_sharpe, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= 8:
            break

    return best_sharpe


## 5. Run Optuna Optimization

In [ ]:
N_TRIALS = 30
bb_tag = "tuned" if TUNE_BACKBONE else BACKBONE
ptp_tag = "_ptp" if USE_PTP else ""
STUDY_NAME = f"mmtfv3_{'_'.join(TICKERS)}_{bb_tag}{ptp_tag}_continuous"

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f"Starting optimization: {N_TRIALS} trials")
print(f"Study: {STUDY_NAME}")
print(f"Tickers: {', '.join(TICKERS)}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return → continuous position")
print(f"Head: {'QuantilePositionHead (5-class PTP)' if USE_PTP else 'Tanh (continuous)'}")
print(f"Loss: {'PTPLoss (CE + PnL + AE)' if USE_PTP else 'ContinuousTradingLoss + SharpeScheduler'}")
print("-" * 60)

In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

In [ ]:
# Best trial results
best_trial = study.best_trial
print(f"\nBest trial #{best_trial.number}:")
print(f"  Sharpe: {best_trial.value:.6f}")
if best_trial.user_attrs:
    attr_keys = ['final_sharpe', 'final_sortino', 'final_win_rate', 'final_dir_acc',
                 'final_pf', 'final_exposure', 'final_downside_vol', 'backbone']
    if USE_PTP:
        attr_keys += ['final_cls_acc', 'final_cls_dir_acc']
    for k in attr_keys:
        print(f"  {k}: {best_trial.user_attrs.get(k, 'N/A')}")

print(f"  Params:")
best_params = best_trial.params
for key, value in sorted(best_params.items()):
    print(f"    {key}: {value}")

# Save
best_params['best_value'] = best_trial.value
best_params['tickers'] = TICKERS
best_params['target_horizon_minutes'] = TARGET_HORIZON_MINUTES
best_params['backbone'] = best_params.get('backbone', BACKBONE)
best_params['use_ptp'] = USE_PTP

prefix = f"{'_'.join(TICKERS)}_mmtfv3_{bb_tag}{ptp_tag}_optuna"
with open(RESULTS_PATH / f"{prefix}_best_params.json", 'w') as f:
    json.dump(best_params, f, indent=2, default=str)
print(f"\nSaved to: {RESULTS_PATH / f'{prefix}_best_params.json'}")

In [ ]:
# Save study artifacts
import joblib

joblib.dump(study, RESULTS_PATH / f"{prefix}_study.pkl")
df_trials = study.trials_dataframe()
df_trials.to_csv(RESULTS_PATH / f"{prefix}_all_trials.csv", index=False)
print(f"Study artifacts saved to {RESULTS_PATH}")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

valid_trials = df_trials[df_trials['state'] == 'COMPLETE']

# 1. Optimization history
ax = axes[0, 0]
ax.plot(valid_trials.index, valid_trials['value'], 'b-o', alpha=0.6, label='Trial Sharpe')
ax.axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Optimization History')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())[:10]
    values = [importances[p] for p in params]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(params)))
    ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except:
    ax.text(0.5, 0.5, 'Not enough completed trials', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance')

# 3. Learning rate vs Sharpe
ax = axes[1, 0]
if 'params_learning_rate' in valid_trials.columns:
    ax.scatter(valid_trials['params_learning_rate'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=100)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Sharpe')
    ax.set_title('Learning Rate vs Sharpe')
    ax.grid(True, alpha=0.3)

# 4. d_model vs Sharpe
ax = axes[1, 1]
if 'params_d_model' in valid_trials.columns:
    d_models = sorted(valid_trials['params_d_model'].unique())
    data_by_d = [valid_trials[valid_trials['params_d_model'] == d]['value'].values for d in d_models]
    bp = ax.boxplot(data_by_d, positions=range(len(d_models)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(d_models)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(d_models)))
    ax.set_xticklabels([str(int(d)) for d in d_models])
    ax.set_xlabel('d_model')
    ax.set_ylabel('Sharpe')
    ax.set_title('Model Size vs Sharpe')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f"MMTFv3 Optimization ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_results.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Loss component analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. tc_cost vs Sharpe
ax = axes[0, 0]
if 'params_tc_cost' in valid_trials.columns:
    ax.scatter(valid_trials['params_tc_cost'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=80)
    ax.set_xscale('log')
    ax.set_xlabel('Transaction Cost')
    ax.set_ylabel('Sharpe')
    ax.set_title('TC Cost vs Sharpe')
    ax.grid(True, alpha=0.3)

# 2. target_exposure vs Sharpe
ax = axes[0, 1]
if 'params_target_exposure' in valid_trials.columns:
    ax.scatter(valid_trials['params_target_exposure'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Target Exposure')
    ax.set_ylabel('Sharpe')
    ax.set_title('Target Exposure vs Sharpe')
    ax.grid(True, alpha=0.3)

# 3. init_direction_weight vs Sharpe
ax = axes[0, 2]
if 'params_init_direction_weight' in valid_trials.columns:
    ax.scatter(valid_trials['params_init_direction_weight'], valid_trials['value'],
               c=valid_trials.index, cmap='coolwarm', alpha=0.7, s=80)
    ax.set_xlabel('Initial Direction Weight')
    ax.set_ylabel('Sharpe')
    ax.set_title('Direction Weight vs Sharpe')
    ax.grid(True, alpha=0.3)

# 4. tech_lookback vs Sharpe
ax = axes[1, 0]
if 'params_tech_lookback' in valid_trials.columns:
    lbs = sorted(valid_trials['params_tech_lookback'].unique())
    data_by_lb = [valid_trials[valid_trials['params_tech_lookback'] == lb]['value'].values for lb in lbs]
    bp = ax.boxplot(data_by_lb, positions=range(len(lbs)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set3(np.linspace(0, 1, len(lbs)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(lbs)))
    ax.set_xticklabels([str(int(lb)) for lb in lbs])
    ax.set_xlabel('Tech Lookback (bars)')
    ax.set_ylabel('Sharpe')
    ax.set_title('Tech Lookback vs Sharpe')
    ax.grid(True, alpha=0.3, axis='y')

# 5. Dropout vs Sharpe
ax = axes[1, 1]
if 'params_dropout' in valid_trials.columns:
    ax.scatter(valid_trials['params_dropout'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Dropout')
    ax.set_ylabel('Sharpe')
    ax.set_title('Dropout vs Sharpe')
    ax.grid(True, alpha=0.3)

# 6. Backbone comparison
ax = axes[1, 2]
if 'params_backbone' in valid_trials.columns:
    for bb_name in ['mamba', 'transformer']:
        mask = valid_trials['params_backbone'] == bb_name
        if mask.any():
            vals = valid_trials.loc[mask, 'value']
            ax.hist(vals, bins=15, alpha=0.5, label=bb_name.upper())
    ax.set_xlabel('Sharpe')
    ax.set_ylabel('Count')
    ax.set_title('Backbone Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Parameter Analysis ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_param_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Train Final Model with Best Parameters

In [ ]:
best = best_params
print("Training final model with best parameters:")
for k, v in sorted(best.items()):
    if k not in ('best_value', 'tickers', 'target_horizon_minutes', 'backbone'):
        print(f"  {k}: {v}")

In [ ]:
# Build final dataloaders + model
tech_lookback = best['tech_lookback']
seq_lookback = best['seq_lookback']

train_loader, val_loader = build_v3_loaders(
    prep,
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    batch_size=best['batch_size'],
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=SAMPLE_STRIDE,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

backbone = best.get('backbone', BACKBONE)
print(f"\nUsing {backbone.upper()} backbone")
print(f"Head: {'QuantilePositionHead (PTP)' if USE_PTP else 'Tanh'}")

base_model = MMTFv3Core(
    f_tech=F_TECH,
    f_seq=F_SEQ,
    f_ae=F_AE,
    ae_type=best.get('ae_type', 'vae'),
    d_latent=best['d_latent'],
    d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'],
    recon_weight=best['recon_weight'],
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'],
    d_static_emb=best['d_static_emb'],
    backbone=backbone,
    n_heads=best['n_heads'],
    n_layers=best['n_layers'],
    d_ff=best.get('d_ff', 512),
    d_state=best.get('d_state', 16),
    d_conv=best.get('d_conv', 4),
    expand=best.get('expand', 2),
    dropout=best['dropout'],
    grn_dropout=best['grn_dropout'],
    numbars_channels=NUMBARS_CHANNELS,
    vpin_channels=VPIN_CHANNELS,
    vpin_bins=VPIN_BINS,
    vpin_time=VPIN_TIME,
)

final_model = StatefulMMTFv3Core(
    base_model=base_model,
    n_tickers=prep.n_tickers,
    quantile_head=USE_PTP,
    ptp_temperature=best.get('ptp_temperature', 1.5),
    state_hidden_dim=best.get('state_hidden_dim', 16),
    state_momentum=best.get('state_momentum', 0.9),
    update_on_eval=True,
).to(device)

print(f"Model parameters: {sum(p.numel() for p in final_model.parameters()):,}")


In [ ]:
NUM_EPOCHS = 30
WARMUP_EPOCHS = 5

trading_kwargs = dict(
    tc_cost=best['tc_cost'],
    direction_weight=best['init_direction_weight'],
    reg_weight=best['init_reg_weight'],
    target_exposure=best['target_exposure'],
    use_sortino=best.get('use_sortino', True),
    downside_vol_weight=best.get('downside_vol_weight', 0.1),
    tc_in_sharpe=best.get('tc_in_sharpe', True),
    holding_weight=best.get('holding_weight', 0.0),
)

if USE_PTP:
    loss_fn = PTPLoss(
        ce_weight=best.get('ce_weight', 1.0),
        pnl_weight=best.get('pnl_weight', 1.0),
        profit_scale=best.get('profit_scale', 100.0),
        inner_threshold=best.get('inner_threshold', 0.25),
        outer_threshold=best.get('outer_threshold', 1.0),
        **trading_kwargs,
    ).to(device)
    _train_fn = train_epoch_v3_ptp
    _eval_fn = evaluate_v3_ptp
    optimizer = optim.AdamW(
        build_ptp_optimizer_param_groups(
            final_model,
            base_lr=best['learning_rate'],
            weight_decay=best['weight_decay'],
            quantile_lr_scale=best.get('quantile_lr_scale', 1.0),
            ptp_lr_scale=best.get('ptp_lr_scale', 1.0),
        )
    )
    lr_scheduler = HeadAwarePTPScheduler(
        optimizer=optimizer,
        total_epochs=NUM_EPOCHS,
        head_decay=best.get('head_lr_decay', 0.6),
        quantile_patience=best.get('quantile_patience', 2),
        ptp_patience=best.get('ptp_patience', 2),
        overfit_tolerance=best.get('head_overfit_tolerance', 0.05),
        ptp_downside_weight=best.get('downside_vol_weight', 0.1),
    )
else:
    loss_fn = ContinuousTradingLoss(**trading_kwargs).to(device)
    _train_fn = train_epoch_v3_stateful
    _eval_fn = evaluate_v3_stateful
    optimizer = optim.AdamW(
        final_model.parameters(),
        lr=best['learning_rate'],
        weight_decay=best['weight_decay'],
    )
    lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=best['learning_rate'] * 0.001,
    )

# SharpeScheduler targets the inner trading loss
_inner_trading_loss = loss_fn.trading_loss if USE_PTP else loss_fn

sharpe_sched = SharpeScheduler(
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    initial_direction_weight=best['init_direction_weight'],
    final_direction_weight=best['final_direction_weight'],
    initial_reg_weight=best['init_reg_weight'],
    final_reg_weight=0.05,
    initial_target_exposure=0.2,
    final_target_exposure=best['target_exposure'],
    initial_holding_weight=0.0,
    final_holding_weight=best.get('holding_weight', 0.0),
)

history = {
    'train_loss': [], 'val_loss': [],
    'val_sharpe': [], 'val_sortino': [],
    'val_win_rate': [], 'val_dir_acc': [],
    'val_pf': [], 'val_exposure': [],
    'val_max_dd': [], 'val_downside_vol': [], 'lr': [],
    'direction_weight': [], 'reg_weight': [],
}
if USE_PTP:
    history['val_cls_acc'] = []
    history['val_ce_loss'] = []
    history['train_ce_loss'] = []
    history['train_trading_loss'] = []
    history['val_trading_loss'] = []
    history['quantile_lr'] = []
    history['ptp_lr'] = []
    history['quantile_action'] = []
    history['ptp_action'] = []

best_sharpe = -1e9
best_state = None

head_tag = "PTP" if USE_PTP else "Tanh"
print(f"Training for {NUM_EPOCHS} epochs (MMTFv3Core + {head_tag} + StateLayer {backbone.upper()})")
print(f"Loss: {'PTPLoss (CE + downside-aware PnL + AE)' if USE_PTP else 'ContinuousTradingLoss'} + SharpeScheduler (warmup={WARMUP_EPOCHS})")
print("=" * 100)

for epoch in range(NUM_EPOCHS):
    sharpe_sched.step(epoch, _inner_trading_loss)

    train_loss, train_metrics = _train_fn(
        final_model, train_loader, loss_fn, optimizer, device,
        max_norm=best['max_norm'], unpack_fn=unpack_v3_batch,
    )
    val_metrics = _eval_fn(
        final_model, val_loader, loss_fn, device,
        unpack_fn=unpack_v3_batch,
    )

    if USE_PTP:
        lr_state = lr_scheduler.step(
            epoch,
            {
                'train_ce_loss': train_metrics.get('ce_loss'),
                'val_ce_loss': val_metrics.get('ce_loss'),
                'train_trading_loss': train_metrics.get('trading_loss'),
                'val_trading_loss': val_metrics.get('trading_loss'),
                'train_downside_vol': train_metrics.get('downside_vol', 0.0),
                'val_downside_vol': val_metrics.get('downside_vol', 0.0),
            },
        )
    else:
        lr_scheduler.step()
        lr_state = {'trunk': optimizer.param_groups[0]['lr']}

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_sharpe'].append(val_metrics['sharpe'])
    history['val_sortino'].append(val_metrics['sortino'])
    history['val_win_rate'].append(val_metrics['win_rate'])
    history['val_dir_acc'].append(val_metrics['dir_accuracy'])
    history['val_pf'].append(val_metrics['profit_factor'])
    history['val_exposure'].append(val_metrics['avg_exposure'])
    history['val_max_dd'].append(val_metrics['max_drawdown'])
    history['val_downside_vol'].append(val_metrics.get('downside_vol', 0.0))
    history['lr'].append(lr_state.get('trunk', optimizer.param_groups[0]['lr']))
    history['direction_weight'].append(_inner_trading_loss.direction_weight)
    history['reg_weight'].append(_inner_trading_loss.reg_weight)
    if USE_PTP:
        history['train_ce_loss'].append(train_metrics.get('ce_loss', 0.0))
        history['train_trading_loss'].append(train_metrics.get('trading_loss', 0.0))
        history['val_trading_loss'].append(val_metrics.get('trading_loss', 0.0))
        history['val_cls_acc'].append(val_metrics.get('cls_accuracy', 0))
        history['val_ce_loss'].append(val_metrics.get('ce_loss', train_metrics.get('ce_loss', 0)))
        history['quantile_lr'].append(lr_state.get('quantile_head', history['lr'][-1]))
        history['ptp_lr'].append(lr_state.get('ptp_head', history['lr'][-1]))
        history['quantile_action'].append(lr_scheduler.last_actions.get('quantile_head', 'n/a'))
        history['ptp_action'].append(lr_scheduler.last_actions.get('ptp_head', 'n/a'))

    is_best = val_metrics['sharpe'] > best_sharpe
    if is_best:
        best_sharpe = val_metrics['sharpe']
        best_state = final_model.state_dict().copy()

    marker = " [BEST]" if is_best else ""
    cls_info = ""
    if USE_PTP and 'cls_accuracy' in val_metrics:
        cls_info = (
            f" | ClsAcc: {val_metrics['cls_accuracy']:.1f}%"
            f" | QLR: {history['quantile_lr'][-1]:.2e}"
            f" | PLR: {history['ptp_lr'][-1]:.2e}"
            f" | DVol: {val_metrics.get('downside_vol', 0.0):.4f}"
        )

    print(
        f"E{epoch+1:02d}/{NUM_EPOCHS} | "
        f"Loss: {train_loss:.4f}/{val_metrics['loss']:.4f} | "
        f"Sharpe: {val_metrics['sharpe']:.4f} | "
        f"Sortino: {val_metrics['sortino']:.4f} | "
        f"WR: {val_metrics['win_rate']:.1f}% | "
        f"PF: {val_metrics['profit_factor']:.2f} | "
        f"Exp: {val_metrics['avg_exposure']:.3f}{cls_info} | "
        f"DW: {_inner_trading_loss.direction_weight:.2f}{marker}"
    )

print("=" * 100)
print(f"Best Sharpe: {best_sharpe:.6f}")


In [ ]:
# Load best model state
if best_state:
    final_model.load_state_dict(best_state)
    if hasattr(final_model, 'reset_position_state'):
        final_model.reset_position_state()
    print("Loaded best model state")



In [ ]:
# Plot training history
n_rows = 3 if USE_PTP else 2
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5 * n_rows))

ax = axes[0, 0]
ax.plot(history['train_loss'], label='Train', alpha=0.8)
ax.plot(history['val_loss'], label='Val', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Total Loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(history['val_sharpe'], 'b-', label='Sharpe', alpha=0.8)
ax.plot(history['val_sortino'], 'g--', label='Sortino', alpha=0.6)
ax.axhline(y=0, color='gray', linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Ratio')
ax.set_title('Risk-Adjusted Returns'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.plot(history['val_win_rate'], 'g-', label='Win Rate', alpha=0.8)
ax.plot(history['val_dir_acc'], 'b--', label='Dir Accuracy', alpha=0.6)
ax.axhline(y=50, color='gray', linestyle=':', label='50%')
ax.set_xlabel('Epoch'); ax.set_ylabel('%')
ax.set_title('Accuracy Metrics'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(history['val_pf'], 'r-', alpha=0.8)
ax.axhline(y=1.0, color='gray', linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Profit Factor')
ax.set_title('Profit Factor'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(history['val_exposure'], 'purple', label='Avg Exposure', alpha=0.8)
ax.plot(history['val_max_dd'], 'red', label='Max Drawdown', alpha=0.6)
ax.plot(history['val_downside_vol'], 'black', linestyle='--', label='Downside Vol', alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('Value')
ax.set_title('Exposure, Drawdown, Downside Vol'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
ax.plot(history['direction_weight'], 'b-', label='Direction Weight', alpha=0.8)
ax.plot(history['reg_weight'], 'r--', label='Reg Weight', alpha=0.6)
ax.set_xlabel('Epoch'); ax.set_ylabel('Weight')
ax.set_title('SharpeScheduler Progression'); ax.legend(); ax.grid(True, alpha=0.3)

if USE_PTP:
    ax = axes[2, 0]
    ax.plot(history['val_cls_acc'], 'darkorange', alpha=0.8)
    ax.axhline(y=20, color='gray', linestyle=':', label='Random (20%)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('%')
    ax.set_title('5-Class Accuracy'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    ax.plot(history['val_ce_loss'], 'crimson', label='Val CE', alpha=0.8)
    ax.plot(history['train_ce_loss'], 'salmon', linestyle='--', label='Train CE', alpha=0.6)
    ax.set_xlabel('Epoch'); ax.set_ylabel('CE Loss')
    ax.set_title('Classification CE Loss'); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[2, 2]
    ax.plot(history['lr'], label='Trunk LR', alpha=0.8)
    ax.plot(history['quantile_lr'], label='Quantile LR', alpha=0.8)
    ax.plot(history['ptp_lr'], label='PTP LR', alpha=0.8)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
    ax.set_yscale('log')
    ax.set_title('Head-Aware LR Schedule'); ax.legend(); ax.grid(True, alpha=0.3)

head_tag = "PTP" if USE_PTP else ""
plt.suptitle(f"MMTFv3 {head_tag} {backbone.upper()} Training ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Model Diagnostics

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import print_v3_diagnostics

# Get tracker from a val pass
final_model.eval()
if hasattr(final_model, 'reset_position_state'):
    final_model.reset_position_state()

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = final_model(
            **inputs, return_ae_losses=True, return_tracker=True,
        )
        # PTP: (position, ae_losses, logits, tracker)
        # Non-PTP: (position, ae_losses, tracker)
        if USE_PTP:
            position, ae_losses, logits, tracker = out
        else:
            position, ae_losses, tracker = out
        break

# Evaluate final metrics
final_metrics = _eval_fn(
    final_model, val_loader, loss_fn, device,
    unpack_fn=unpack_v3_batch,
)

print_v3_diagnostics(tracker, final_metrics, epoch=NUM_EPOCHS)
print(
    f"State layer -> prev_state_mean: {tracker.get('avg_prev_state', 0.0):.4f}, "
    f"delta_mean: {tracker.get('avg_state_delta', 0.0):.4f}"
)

if USE_PTP:
    print(f"\nPTP Classification Metrics:")
    print(f"  5-class accuracy: {final_metrics.get('cls_accuracy', 0):.1f}%")
    for c in range(5):
        acc_key = f'cls_{c}_acc'
        cnt_key = f'cls_{c}_count'
        if acc_key in final_metrics:
            print(f"  Class {c}: {final_metrics[acc_key]:.1f}% (n={final_metrics.get(cnt_key, 0)})")


In [ ]:
# Position distribution analysis
final_model.eval()
if hasattr(final_model, 'reset_position_state'):
    final_model.reset_position_state()

all_positions = []
all_returns = []
all_pred_classes = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = final_model(**inputs, return_ae_losses=True)
        if USE_PTP:
            position, _, logits = out
            all_pred_classes.append(logits.argmax(dim=-1).cpu().numpy())
        else:
            position, _ = out
        all_positions.append(position.view(-1).cpu().numpy())
        all_returns.append(targets.view(-1).cpu().numpy())

positions = np.concatenate(all_positions)
returns = np.concatenate(all_returns)

n_cols = 4 if USE_PTP else 3
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))

# Position distribution
ax = axes[0]
ax.hist(positions, bins=100, alpha=0.7, color='steelblue')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Position')
ax.set_ylabel('Count')
ax.set_title(f'Position Distribution (mean={positions.mean():.3f}, std={positions.std():.3f})')
ax.grid(True, alpha=0.3)

# Position vs return scatter
ax = axes[1]
ax.scatter(returns, positions, alpha=0.05, s=5, c='steelblue')
ax.axhline(y=0, color='gray', linestyle=':')
ax.axvline(x=0, color='gray', linestyle=':')
ax.set_xlabel('Forward Return')
ax.set_ylabel('Position')
ax.set_title('Position vs Return')
ax.grid(True, alpha=0.3)

# Cumulative strategy PnL
strategy_ret = positions * returns
cum_pnl = np.cumsum(strategy_ret)
ax = axes[2]
ax.plot(cum_pnl, 'b-', alpha=0.8)
ax.axhline(y=0, color='gray', linestyle=':')
ax.fill_between(range(len(cum_pnl)), cum_pnl, 0,
                where=cum_pnl >= 0, color='green', alpha=0.1)
ax.fill_between(range(len(cum_pnl)), cum_pnl, 0,
                where=cum_pnl < 0, color='red', alpha=0.1)
ax.set_xlabel('Sample')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Validation Cumulative PnL')
ax.grid(True, alpha=0.3)

# PTP: predicted class distribution
if USE_PTP:
    pred_classes = np.concatenate(all_pred_classes)
    ax = axes[3]
    class_names = ['Strong-', 'Weak-', 'Neutral', 'Weak+', 'Strong+']
    colors = ['#d62728', '#ff7f0e', '#7f7f7f', '#2ca02c', '#1f77b4']
    counts = [np.sum(pred_classes == c) for c in range(5)]
    ax.bar(range(5), counts, color=colors, alpha=0.8)
    ax.set_xticks(range(5))
    ax.set_xticklabels(class_names, rotation=30)
    ax.set_ylabel('Count')
    ax.set_title('Predicted Class Distribution')
    ax.grid(True, alpha=0.3, axis='y')

head_tag = "PTP " if USE_PTP else ""
plt.suptitle(f"MMTFv3 {head_tag}{backbone.upper()} Position Analysis", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_position_analysis.png", dpi=150, bbox_inches='tight')
plt.show()


## 9. Save Final Model

In [ ]:
model_path = RESULTS_PATH / f"{prefix}_best_model.pth"

save_dict = {
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_sharpe': best_sharpe,
    'params': best,
    'tickers': TICKERS,
    'dims': dims,
    'n_tickers': prep.n_tickers,
    'n_asset_classes': prep.n_asset_classes,
    'n_asset_subclasses': prep.n_asset_subclasses,
    'target_horizon_minutes': TARGET_HORIZON_MINUTES,
    'history': history,
    'final_metrics': final_metrics,
    'architecture': 'MMTFv3Core+PTP+StateLayer' if USE_PTP else 'MMTFv3Core+StateLayer',
    'backbone': backbone,
    'use_ptp': USE_PTP,
    'state_config': {
        'state_hidden_dim': best.get('state_hidden_dim', 16),
        'state_momentum': best.get('state_momentum', 0.9),
        'update_on_eval': True,
    },
    'ae_config': {
        'f_ae': F_AE,
        'ae_type': best.get('ae_type', 'vae'),
        'd_latent': best['d_latent'],
        'd_ae_hidden': best['d_ae_hidden'],
        'kl_weight': best['kl_weight'],
        'recon_weight': best['recon_weight'],
        'ae_features': ['return_1d', 'return_5d', 'return_21d', 'rv_1d'],
    },
    'tech_feature_cols': prep._tech_feature_cols,
}

if USE_PTP:
    save_dict['ptp_config'] = {
        'ptp_temperature': best.get('ptp_temperature', 1.5),
        'ce_weight': best.get('ce_weight', 1.0),
        'pnl_weight': best.get('pnl_weight', 1.0),
        'profit_scale': best.get('profit_scale', 100.0),
        'inner_threshold': best.get('inner_threshold', 0.25),
        'outer_threshold': best.get('outer_threshold', 1.0),
        'quantile_lr_scale': best.get('quantile_lr_scale', 1.0),
        'ptp_lr_scale': best.get('ptp_lr_scale', 1.0),
        'head_lr_decay': best.get('head_lr_decay', 0.6),
        'quantile_patience': best.get('quantile_patience', 2),
        'ptp_patience': best.get('ptp_patience', 2),
        'head_overfit_tolerance': best.get('head_overfit_tolerance', 0.05),
    }
    save_dict['loss_config'] = {
        'type': 'PTPLoss',
        'ce_weight': best.get('ce_weight', 1.0),
        'pnl_weight': best.get('pnl_weight', 1.0),
        'tc_cost': best['tc_cost'],
        'use_sortino': best.get('use_sortino', True),
        'downside_vol_weight': best.get('downside_vol_weight', 0.1),
        'scheduler': 'SharpeScheduler + HeadAwarePTPScheduler',
    }
else:
    save_dict['loss_config'] = {
        'type': 'ContinuousTradingLoss',
        'tc_cost': best['tc_cost'],
        'use_sortino': best.get('use_sortino', True),
        'downside_vol_weight': best.get('downside_vol_weight', 0.1),
        'scheduler': 'SharpeScheduler',
    }

torch.save(save_dict, model_path)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_PATH / f"{prefix}_training_history.csv", index=False)

head_tag = "PTP + " if USE_PTP else ""
print(f"\n{'=' * 60}")
print('TRAINING COMPLETE')
print(f"{'=' * 60}")
print(f"\nArchitecture: MMTFv3Core + {head_tag}StateLayer {backbone.upper()}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return -> position in [-1, 1]")
print(f"AE input: [ret_1d, ret_5d, ret_21d, rv_1d] (f_ae={F_AE})")
if USE_PTP:
    print(f"Head: QuantilePositionHead (5-class -> context-aware PTP -> continuous)")
    print(f"Loss: PTPLoss (CE={best.get('ce_weight', 1.0):.2f} + PnL={best.get('pnl_weight', 1.0):.2f} + AE)")
    print(f"Downside vol weight: {best.get('downside_vol_weight', 0.1):.4f}")
print(f"Best Sharpe: {best_sharpe:.6f}")
if USE_PTP and 'cls_accuracy' in final_metrics:
    print(f"Final 5-class accuracy: {final_metrics['cls_accuracy']:.1f}%")
print(f"Final downside vol: {final_metrics.get('downside_vol', 0.0):.6f}")
print(f"\nArtifacts saved to: {RESULTS_PATH}")
print(f"  - {prefix}_best_model.pth")
print(f"  - {prefix}_best_params.json")
print(f"  - {prefix}_study.pkl")
print(f"  - {prefix}_all_trials.csv")
print(f"  - {prefix}_results.png")
print(f"  - {prefix}_training_history.png")
print(f"  - {prefix}_position_analysis.png")


## 10. Validation Backtest

In [ ]:
# ── Validation Backtest ───────────────────────────────────────────────────────
# Runs the best model on validation data and reports a full suite of trading
# performance statistics (Sharpe, Sortino, net PnL, drawdown, win rate, …)
# broken down per ticker and combined.
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

final_model.eval()
if hasattr(final_model, "reset_position_state"):
    final_model.reset_position_state()

_bt_pos, _bt_ret, _bt_tid = [], [], []
with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = final_model(**inputs, return_ae_losses=True)
        if USE_PTP:
            position, _, _logits = out
        else:
            position, _ = out
        _bt_pos.append(position.view(-1).cpu())
        _bt_ret.append(targets.view(-1).float().cpu())
        _bt_tid.append(inputs["ticker_id"].view(-1).cpu())

bt_pos = torch.cat(_bt_pos).numpy()   # (N,)  positions in [-1, 1]
bt_ret = torch.cat(_bt_ret).numpy()   # (N,)  forward log-returns
bt_tid = torch.cat(_bt_tid).numpy()   # (N,)  integer ticker ids

# id → ticker name (prep.registry built from sorted(TICKERS))
id_to_ticker = {meta.ticker_id: t for t, meta in prep.registry.items()}

# ── Stat helper ───────────────────────────────────────────────────────────────
_TRADE_THRESH = 0.05   # |position| > threshold → "active bar"

def _bt_stats(pos, ret, label):
    sr = pos * ret
    cum = np.cumsum(sr)
    non_flat = np.abs(pos) > _TRADE_THRESH
    gp = sr[sr > 0].sum()
    gl = np.abs(sr[sr < 0]).sum() + 1e-9
    correct = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc  = (correct & non_flat).sum() / max(non_flat.sum(), 1)
    win_rate = ((sr[non_flat] > 0).mean() * 100) if non_flat.sum() > 0 else 0.0
    run_max  = np.maximum.accumulate(cum)
    mdd      = float((run_max - cum).max())
    mean_sr  = sr.mean()
    std_sr   = sr.std() + 1e-8
    # Sortino: penalise only downside semi-deviation
    neg_sr   = sr[sr < 0]
    dside    = float(np.sqrt((neg_sr ** 2).mean())) if len(neg_sr) > 0 else 1e-8
    # # Trades = number of sign-change events in the position series
    signs    = np.sign(pos)
    n_trades = int((np.diff(signs) != 0).sum())
    return {
        "Ticker":          label,
        "Net PnL":         round(float(sr.sum()), 6),
        "Sharpe":          round(float(mean_sr / std_sr), 4),
        "Sortino":         round(float(mean_sr / (dside + 1e-8)), 4),
        "Win Rate (%)":    round(float(win_rate), 2),
        "Dir Acc (%)":     round(float(dir_acc * 100), 2),
        "Profit Factor":   round(float(gp / gl), 4),
        "Max Drawdown":    round(mdd, 6),
        "# Trades":        n_trades,
        "# Active Bars":   int(non_flat.sum()),
        "Avg |Position|":  round(float(np.abs(pos).mean()), 4),
        "N Samples":       len(pos),
    }

# ── Build results table ───────────────────────────────────────────────────────
rows = []
for tid in sorted(id_to_ticker):
    ticker = id_to_ticker[tid]
    mask   = bt_tid == tid
    if mask.sum() == 0:
        continue
    rows.append(_bt_stats(bt_pos[mask], bt_ret[mask], ticker))

rows.append(_bt_stats(bt_pos, bt_ret, "COMBINED"))

df_bt = pd.DataFrame(rows).set_index("Ticker")

print("=" * 75)
print("VALIDATION BACKTEST  —  MMTFv3 Best Model")
print(f"Tickers : {', '.join(TICKERS)}  |  "
      f"Target  : {TARGET_HORIZON_MINUTES}min forward return  |  "
      f"Samples : {len(bt_pos):,}")
print("=" * 75)
print(df_bt.to_string())
print("=" * 75)
print("Sharpe/Sortino are per-bar (not annualised).  "
      "# Trades = direction-change events in the position series.")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
palette = plt.cm.tab10(np.linspace(0, 0.4, len(TICKERS)))

comb_pnl = np.cumsum(bt_pos * bt_ret)

# 1. Cumulative PnL — per-ticker + combined
ax = axes[0, 0]
for tid, color in zip(sorted(id_to_ticker), palette):
    ticker = id_to_ticker[tid]
    mask   = bt_tid == tid
    sr_t   = (bt_pos * bt_ret)[mask]
    ax.plot(np.cumsum(sr_t), label=ticker, color=color, lw=1.5, alpha=0.85)
ax.plot(comb_pnl, label="Combined", color="black", lw=2, linestyle="--")
ax.axhline(0, color="gray", linestyle=":", lw=0.8)
ax.fill_between(range(len(comb_pnl)), comb_pnl, 0,
                where=comb_pnl >= 0, color="green", alpha=0.08)
ax.fill_between(range(len(comb_pnl)), comb_pnl, 0,
                where=comb_pnl < 0, color="red", alpha=0.08)
ax.set_title("Cumulative PnL — Validation")
ax.set_xlabel("Sample")
ax.set_ylabel("Cumulative PnL")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Net PnL bar chart by ticker (excl. COMBINED row)
ax = axes[0, 1]
tk_labels = [r["Ticker"]  for r in rows[:-1]]
pnl_vals  = [r["Net PnL"] for r in rows[:-1]]
bar_cols   = ["#27ae60" if v >= 0 else "#e74c3c" for v in pnl_vals]
ax.bar(tk_labels, pnl_vals, color=bar_cols, alpha=0.85, edgecolor="white")
_offset = max(abs(v) for v in pnl_vals) * 0.03 if pnl_vals else 0
for i, v in enumerate(pnl_vals):
    ax.text(i, v + (1 if v >= 0 else -1) * _offset,
            f"{v:.4f}", ha="center",
            va="bottom" if v >= 0 else "top", fontsize=9)
ax.axhline(0, color="gray", linestyle=":", lw=0.8)
ax.set_title("Net PnL by Ticker")
ax.set_ylabel("Net PnL")
ax.grid(True, alpha=0.3, axis="y")

# 3. Sharpe & Sortino grouped bar — per ticker
ax = axes[1, 0]
x = np.arange(len(TICKERS))
w = 0.35
sharpe_vals  = [r["Sharpe"]  for r in rows[:-1]]
sortino_vals = [r["Sortino"] for r in rows[:-1]]
ax.bar(x - w / 2, sharpe_vals,  w, label="Sharpe",  color="steelblue",  alpha=0.85)
ax.bar(x + w / 2, sortino_vals, w, label="Sortino", color="darkorange", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(TICKERS)
ax.axhline(0, color="gray", linestyle=":", lw=0.8)
ax.set_title("Sharpe & Sortino by Ticker (per-bar)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# 4. Rolling drawdown curve — combined
ax = axes[1, 1]
run_max  = np.maximum.accumulate(comb_pnl)
dd_curve = run_max - comb_pnl
ax.fill_between(range(len(dd_curve)), dd_curve, color="#e74c3c", alpha=0.45)
ax.plot(dd_curve, color="#c0392b", lw=0.8, alpha=0.8)
ax.set_title(f"Drawdown — Combined  (Max: {dd_curve.max():.5f})")
ax.set_xlabel("Sample")
ax.set_ylabel("Drawdown")
ax.grid(True, alpha=0.3)

plt.suptitle(
    f"MMTFv3 Validation Backtest — {', '.join(TICKERS)}",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_val_backtest.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. Detailed Portfolio Backtest — Branch Weights & Close Charts

In [ ]:
# ── Detailed Portfolio Backtest ──────────────────────────────────────────────
# Rebuilds the validation split (same date-cutoff logic as build_v3_loaders),
# runs forward passes with return_tracker=True to capture per-sample branch
# weights, and aligns results to calendar dates for plotting.
# ─────────────────────────────────────────────────────────────────────────────
from torch.utils.data import DataLoader as _DL
from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousDataset,
    v3_collate_fn,
)

VAL_RATIO = 0.2     # must match build_v3_loaders default
BT_STRIDE = SAMPLE_STRIDE
BT_BATCH  = 64

# 1. Rebuild all samples to recover per-sample metadata (date, ticker)
print("Rebuilding samples for backtest …")
all_bt_samples = prep.build_samples(
    tech_lookback=best_params["tech_lookback"],
    seq_lookback_bars=best_params["seq_lookback"],
    session_only=True,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=BT_STRIDE,
)
print(f"  Total samples: {len(all_bt_samples):,}")

# 2. Replicate the same train/val cutoff
_all_dates  = sorted(set(s["date"] for s in all_bt_samples))
_n_val      = max(1, int(len(_all_dates) * VAL_RATIO))
_bt_cutoff  = _all_dates[-_n_val]
print(f"  Val cutoff: {_bt_cutoff}  ({_n_val} days)")

bt_val_samples = [s for s in all_bt_samples if s["date"] >= _bt_cutoff]
bt_meta        = [{"ticker": s["ticker"], "date": s["date"]}
                  for s in bt_val_samples]
print(f"  Val samples: {len(bt_val_samples):,}")

# 3. Build loader – no shuffle so sample index aligns with bt_meta
_bt_ds = V3ContinuousDataset(
    bt_val_samples,
    profile_shape=prep.profile_shape,
    raster_shape=prep.raster_shape,
)
_bt_loader = _DL(
    _bt_ds, batch_size=BT_BATCH, shuffle=False,
    collate_fn=v3_collate_fn, num_workers=0,
)

# 4. Forward pass – collect position, return, branch weights
final_model.eval()
if hasattr(final_model, "reset_position_state"):
    final_model.reset_position_state()

_all_pos = []
_all_ret = []
_all_bw  = {"backbone_fused": [], "spatial": [], "sequential": []}

with torch.no_grad():
    for batch in _bt_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        B = targets.shape[0]

        out = final_model(**inputs, return_ae_losses=True, return_tracker=True)
        if USE_PTP:
            position, _ae, _logits, tracker = out
        else:
            position, _ae, tracker = out

        _all_pos.append(position.view(-1).cpu().numpy())
        _all_ret.append(targets.view(-1).float().cpu().numpy())

        bw = tracker.get("branch_weights", {})
        for bname in _all_bw:
            val_bw = bw.get(bname, float("nan"))
            _all_bw[bname].extend([val_bw] * B)   # batch-level mean → per sample

# 5. Build flat result DataFrame with metadata
bt_df = pd.DataFrame({
    "date":          [m["date"]   for m in bt_meta],
    "ticker":        [m["ticker"] for m in bt_meta],
    "position":      np.concatenate(_all_pos),
    "return":        np.concatenate(_all_ret),
    "pnl":           np.concatenate(_all_pos) * np.concatenate(_all_ret),
    "bw_backbone":   _all_bw["backbone_fused"],
    "bw_spatial":    _all_bw["spatial"],
    "bw_sequential": _all_bw["sequential"],
})
bt_df["date"] = pd.to_datetime(bt_df["date"])

print(f"\nBacktest DataFrame shape: {bt_df.shape}")
print(bt_df.head(3))


In [ ]:
# ── Portfolio Statistics ──────────────────────────────────────────────────────
_THRESH = 0.05   # |pos| > threshold → "active bar"

# Bars per year: 6.5h session × 12 five-min bars/h × 252 trading days
_BARS_PER_YEAR = 252 * 78


def portfolio_stats(pos, ret, label=""):
    sr      = pos * ret
    cum     = np.cumsum(sr)
    active  = np.abs(pos) > _THRESH
    gp      = sr[sr > 0].sum()
    gl      = np.abs(sr[sr < 0]).sum() + 1e-12

    correct  = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc  = correct[active].mean() * 100 if active.sum() > 0 else 0.0
    win_rate = (sr[active] > 0).mean()  * 100 if active.sum() > 0 else 0.0

    mu    = sr.mean()
    sigma = sr.std() + 1e-12
    neg   = sr[sr < 0]
    dside = float(np.sqrt((neg ** 2).mean())) if len(neg) > 0 else 1e-12
    run_max  = np.maximum.accumulate(cum)
    mdd      = float((run_max - cum).max())
    calmar   = float(cum[-1]) / (mdd + 1e-12)
    sharpe_ann = (mu / sigma) * np.sqrt(_BARS_PER_YEAR)

    signs    = np.sign(pos)
    n_trades = int((np.diff(signs) != 0).sum())

    return {
        "Label":              label,
        "Net PnL":            round(float(sr.sum()), 6),
        "Sharpe (per-bar)":   round(float(mu / sigma), 5),
        "Sharpe (ann.)":      round(float(sharpe_ann), 3),
        "Sortino (per-bar)":  round(float(mu / dside), 5),
        "Calmar":             round(float(calmar), 4),
        "Win Rate (%)":       round(float(win_rate), 2),
        "Dir Acc (%)":        round(float(dir_acc), 2),
        "Profit Factor":      round(float(gp / gl), 4),
        "Max Drawdown":       round(float(mdd), 6),
        "# Trades":           n_trades,
        "# Active Bars":      int(active.sum()),
        "Avg |Position|":     round(float(np.abs(pos).mean()), 4),
        "N Samples":          len(pos),
    }


stat_rows = []
for ticker in sorted(bt_df["ticker"].unique()):
    mask = bt_df["ticker"] == ticker
    stat_rows.append(
        portfolio_stats(
            bt_df.loc[mask, "position"].values,
            bt_df.loc[mask, "return"].values,
            label=ticker,
        )
    )

# Combined portfolio (all positions + returns concatenated)
stat_rows.append(
    portfolio_stats(
        bt_df["position"].values,
        bt_df["return"].values,
        label="PORTFOLIO",
    )
)

df_stats = pd.DataFrame(stat_rows).set_index("Label")

_line = "=" * 88
print(_line)
print("  DETAILED BACKTEST — MMTFv3  |  Val start:", str(_bt_cutoff))
print(f"  Tickers: {', '.join(TICKERS)}   "
      f"Target: {TARGET_HORIZON_MINUTES}min   Samples: {len(bt_df):,}")
print(_line)
print(df_stats.T.to_string())
print(_line)

port_sharpe = df_stats.loc["PORTFOLIO", "Sharpe (ann.)"]
print(f"\n{'*' * 42}")
print(f"  Portfolio Sharpe (ann.) = {port_sharpe:.3f}")
print(f"{'*' * 42}")


In [ ]:
# ── Load Close Prices from intraday.csv ───────────────────────────────────────
from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df

close_dfs = {}
for ticker in TICKERS:
    _path = DATA_ROOT / ticker / "intraday.csv"
    _raw  = read_exported_df(str(_path))
    if not isinstance(_raw.index, pd.DatetimeIndex):
        _raw.index = pd.to_datetime(_raw.index)
    _close_col = next(
        (c for c in _raw.columns if c.lower() in ("close", "last")),
        _raw.columns[0],
    )
    _raw = _raw[[_close_col]].rename(columns={_close_col: "close"})
    _daily = _raw["close"].resample("1D").last().dropna()
    _daily.index = _daily.index.normalize()
    close_dfs[ticker] = _daily
    print(f"  [{ticker}] {len(_raw):,} bars → {len(_daily)} daily closes  "
          f"({_daily.index[0].date()} – {_daily.index[-1].date()})")


In [ ]:
# ── Per-Ticker Composite Chart ────────────────────────────────────────────────
# Each ticker gets a 3-row figure (shared x-axis):
#   Row 1: Close price (from intraday.csv, daily)
#   Row 2: Branch weights — backbone_fused / spatial / sequential
#   Row 3: Cumulative PnL (daily, log-return units)
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.dates as mdates

_BW_META = [
    ("bw_backbone",   "backbone_fused", "#1f77b4"),
    ("bw_spatial",    "spatial",        "#ff7f0e"),
    ("bw_sequential", "sequential",     "#2ca02c"),
]

for ticker in sorted(bt_df["ticker"].unique()):
    tmask = bt_df["ticker"] == ticker
    t_df  = bt_df[tmask].copy()

    # Daily aggregates
    daily = (
        t_df.groupby("date")
        .agg(
            day_pnl      = ("pnl",          "sum"),
            bw_backbone  = ("bw_backbone",   "mean"),
            bw_spatial   = ("bw_spatial",    "mean"),
            bw_sequential= ("bw_sequential", "mean"),
        )
        .sort_index()
    )
    daily["cum_pnl"] = daily["day_pnl"].cumsum()

    _close = close_dfs.get(ticker, pd.Series(dtype=float))
    _close = _close.reindex(daily.index, method="ffill")

    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(16, 11),
        sharex=True,
        gridspec_kw={"height_ratios": [2, 1.5, 1.5]},
    )
    fig.suptitle(
        f"MMTFv3 Backtest — {ticker}  |  "
        f"Val: {daily.index[0].date()} → {daily.index[-1].date()}",
        fontsize=13, fontweight="bold",
    )

    # Row 1: Close price
    if _close.notna().any():
        ax1.plot(_close.index, _close.values, color="#222222", lw=1.5,
                 label="Close (daily)")
        ax1.fill_between(
            _close.index, _close.values, _close.values.min(),
            alpha=0.08, color="#222222",
        )
    else:
        ax1.text(0.5, 0.5, "No close data in window",
                 transform=ax1.transAxes, ha="center", va="center", color="gray")
    ax1.set_ylabel("Price", fontsize=10)
    ax1.set_title(f"{ticker} — Close Price (intraday.csv → daily last)",
                  fontsize=10, pad=3)
    ax1.legend(loc="upper left", fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v:,.2f}")
    )

    # Row 2: Branch weights
    for col, label, color in _BW_META:
        if col in daily.columns:
            ax2.plot(daily.index, daily[col], label=label,
                     color=color, lw=1.5, alpha=0.85)
    ax2.set_ylim(bottom=0)
    ax2.set_ylabel("Branch Weight", fontsize=10)
    ax2.set_title(
        f"{ticker} — Regime-Conditioned Branch Weights (BVS avg. per day)",
        fontsize=10, pad=3,
    )
    ax2.legend(loc="upper left", fontsize=8, ncol=3)
    ax2.grid(True, alpha=0.3)

    # Row 3: Cumulative PnL
    ax3.plot(daily.index, daily["cum_pnl"], color="#1a6bbf", lw=1.8,
             label="Cum. PnL")
    ax3.fill_between(
        daily.index, daily["cum_pnl"], 0,
        where=daily["cum_pnl"] >= 0,
        color="#27ae60", alpha=0.15, label="Profit zone",
    )
    ax3.fill_between(
        daily.index, daily["cum_pnl"], 0,
        where=daily["cum_pnl"] < 0,
        color="#e74c3c", alpha=0.15, label="Loss zone",
    )
    ax3.axhline(0, color="gray", linestyle="--", lw=0.8)
    ax3.set_ylabel("Cumul. PnL\n(log-ret)", fontsize=10)
    ax3.set_title(f"{ticker} — Cumulative PnL", fontsize=10, pad=3)
    ax3.legend(loc="upper left", fontsize=8, ncol=3)
    ax3.grid(True, alpha=0.3)

    # Annotate trade stats
    _ts = df_stats.loc[ticker] if ticker in df_stats.index else {}
    _ann = (
        f"Sharpe (ann.) = {_ts.get('Sharpe (ann.)', float('nan')):.3f}  |  "
        f"Sortino = {_ts.get('Sortino (per-bar)', float('nan')):.4f}  |  "
        f"Net PnL = {_ts.get('Net PnL', float('nan')):.4f}  |  "
        f"Win Rate = {_ts.get('Win Rate (%)', float('nan')):.1f}%  |  "
        f"Max DD = {_ts.get('Max Drawdown', float('nan')):.4f}  |  "
        f"# Trades = {_ts.get('# Trades', 'N/A')}"
    )
    ax3.annotate(
        _ann,
        xy=(0.01, 0.04), xycoords="axes fraction",
        fontsize=7.5, color="#333333",
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.85),
    )

    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)

    plt.tight_layout()
    _out = RESULTS_PATH / f"{prefix}_backtest_{ticker}.png"
    plt.savefig(_out, dpi=150, bbox_inches="tight")
    print(f"Saved: {_out}")
    plt.show()


In [ ]:
# ── Combined Portfolio Summary Chart ─────────────────────────────────────────
import matplotlib.dates as _mdates

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

_palette = {t: c for t, c in zip(
    sorted(bt_df["ticker"].unique()),
    plt.cm.tab10(np.linspace(0, 0.35, len(TICKERS))),
)}

# Per-ticker + combined cumulative PnL
ax = axes[0, 0]
combined_cum = bt_df.groupby("date")["pnl"].sum().cumsum()
for ticker in sorted(bt_df["ticker"].unique()):
    tmask = bt_df["ticker"] == ticker
    tk_cum = bt_df[tmask].groupby("date")["pnl"].sum().cumsum()
    ax.plot(tk_cum.index, tk_cum.values,
            color=_palette[ticker], lw=1.5, label=ticker, alpha=0.85)
ax.plot(combined_cum.index, combined_cum.values,
        color="black", lw=2.2, linestyle="--", label="Portfolio")
ax.axhline(0, color="gray", lw=0.8, linestyle=":")
ax.fill_between(combined_cum.index, combined_cum.values, 0,
                where=combined_cum.values >= 0, color="green", alpha=0.07)
ax.fill_between(combined_cum.index, combined_cum.values, 0,
                where=combined_cum.values < 0, color="red", alpha=0.07)
ax.set_title("Cumulative PnL — Per-Ticker & Portfolio")
ax.set_ylabel("Cumulative PnL (log-ret)")
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(_mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(_mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)

# Annualised Sharpe by ticker
ax = axes[0, 1]
_tickers_only = [l for l in df_stats.index if l != "PORTFOLIO"]
_sharpe_vals  = [df_stats.loc[t, "Sharpe (ann.)"] for t in _tickers_only]
_bar_cols = ["#27ae60" if v >= 0 else "#e74c3c" for v in _sharpe_vals]
_bars = ax.bar(_tickers_only, _sharpe_vals, color=_bar_cols,
               alpha=0.85, edgecolor="white")
ax.axhline(0, color="gray", lw=0.8, linestyle=":")
for bar, v in zip(_bars, _sharpe_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v + (0.02 if v >= 0 else -0.04),
        f"{v:.3f}", ha="center",
        va="bottom" if v >= 0 else "top",
        fontsize=9, fontweight="bold",
    )
ax.set_title("Annualised Sharpe by Ticker")
ax.set_ylabel("Sharpe (ann.)")
ax.grid(True, alpha=0.3, axis="y")

# Average branch weights by ticker (grouped bar)
ax = axes[1, 0]
_bw_agg = (
    bt_df.groupby("ticker")[["bw_backbone", "bw_spatial", "bw_sequential"]]
    .mean()
    .rename(columns={
        "bw_backbone":   "backbone_fused",
        "bw_spatial":    "spatial",
        "bw_sequential": "sequential",
    })
)
_x     = np.arange(len(_bw_agg))
_width = 0.25
_bw_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for i, (col, color) in enumerate(zip(_bw_agg.columns, _bw_colors)):
    ax.bar(_x + i * _width, _bw_agg[col], _width,
           label=col, color=color, alpha=0.85)
ax.set_xticks(_x + _width)
ax.set_xticklabels(_bw_agg.index)
ax.set_title("Mean Branch Weights by Ticker (BVS)")
ax.set_ylabel("Avg. Weight")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# Portfolio drawdown
ax = axes[1, 1]
_port_vals = combined_cum.values
_run_max   = np.maximum.accumulate(_port_vals)
_dd        = _run_max - _port_vals
ax.fill_between(combined_cum.index, _dd, color="#e74c3c", alpha=0.45)
ax.plot(combined_cum.index, _dd, color="#c0392b", lw=0.9)
ax.set_title(f"Portfolio Drawdown  (Max: {_dd.max():.5f})")
ax.set_ylabel("Drawdown")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(_mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(_mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)

plt.suptitle(
    f"MMTFv3 Portfolio Backtest — {', '.join(TICKERS)}  |  "
    f"Portfolio Sharpe (ann.) = {df_stats.loc['PORTFOLIO', 'Sharpe (ann.)']:.3f}",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
_out = RESULTS_PATH / f"{prefix}_backtest_portfolio.png"
plt.savefig(_out, dpi=150, bbox_inches="tight")
print(f"Saved: {_out}")
plt.show()

# ── Full trade statistics table ────────────────────────────────────────────────
print("\n" + "=" * 90)
print("  FULL TRADE STATISTICS")
print("=" * 90)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
print(df_stats.to_string())
print("=" * 90)
